In [2]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, MessagesState
from langgraph.graph import START, END
from llm_factory import LLMFactory
from langchain_core.messages import SystemMessage, trim_messages
from typing import TypedDict  

C:\davi_tonon\mestrado\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [3]:
from config import DATA_DIR
files = [

 DATA_DIR / 'Microsoft365DefenderEvents.json',

]

from config import DATA_DIR

print(files)

import json
data = []
with open(files[0], 'r') as f:
    #first_char = f.read(1)
    #f.seek(0)
    print(f)
    data = json.load(f)
print(type(data))
print(f"Total de eventos: {len(data)}")
print(f"Exemplo:\n {data[0]}")

[WindowsPath('C:/davi_tonon/mestrado/data/raw/Microsoft365DefenderEvents.json')]
<_io.TextIOWrapper name='C:\\davi_tonon\\mestrado\\data\\raw\\Microsoft365DefenderEvents.json' mode='r' encoding='cp1252'>
<class 'list'>
Total de eventos: 9
Exemplo:
 {'Timestamp': '2021-08-02T13:11:39.7724491Z', 'DeviceId': '00000000-0000-0000-0000-000000000000', 'DeviceName': 'adfs01.simulandlabs.com', 'ActionType': 'LdapSearch', 'FileName': '', 'FolderPath': '', 'SHA1': '', 'SHA256': '', 'MD5': '', 'FileSize': None, 'AccountDomain': '', 'AccountName': '', 'AccountSid': '', 'RemoteUrl': '', 'RemoteDeviceName': '', 'ProcessId': None, 'ProcessCommandLine': '', 'ProcessCreationTime': None, 'ProcessTokenElevation': '', 'LogonId': None, 'RegistryKey': '', 'RegistryValueName': '', 'RegistryValueData': '', 'RemoteIP': '', 'RemotePort': None, 'LocalIP': '', 'LocalPort': None, 'FileOriginUrl': '', 'FileOriginIP': '', 'InitiatingProcessSHA1': '6cbce4a295c163791b60fc23d285e6d84f28ee4c', 'InitiatingProcessSHA256': 

In [4]:
SYSTEM_PROMPT = """
You are a cybersecurity analyst specialized in log analysis and MITRE ATT&CK mapping.
You will receive many event of a file. Analyze each event carefully.

You are able to:
- Detect anomalies in logs
- Identify security events
- Map findings to MITRE ATT&CK TTPs

Rules:
- Always base your analysis on evidence
- Do NOT hallucinate
- Be precise and technical
- Always answer in English

### OUTPUT REQUIREMENTS
Provide the results in the following structured format:

**MITRE ATT&CK Mapping**
- **Tactics:**
- **Techniques:**
- **Technique IDs:**



"""

In [6]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.graph import MessagesState  # ← Importante!
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

# ❌ NÃO use TypedDict customizado para messages
# ✅ Use MessagesState que já sabe acumular

def analyze_logs(state: MessagesState):
    llm = LLMFactory().get_model()
    llm = ChatOllama(model="llama3.1:8b", temperature=0.1)
    
    # MessagesState já tem 'messages' como chave
    response = llm.invoke(state["messages"])
    
    # Retorna a nova mensagem (MessagesState acumula automaticamente)
    return {"messages": [response]}

builder = StateGraph(MessagesState)  # ← MessagesState aqui
builder.add_node("analyze", analyze_logs)
builder.add_edge(START, "analyze")
builder.add_edge("analyze", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "sessao-teste"}}

# Teste
result1 = graph.invoke(
    {"messages": [HumanMessage(content=SYSTEM_PROMPT)]},
    config
)
print("Resposta 1:", result1["messages"][-1].content)


result2 = graph.invoke(
        {"messages": [HumanMessage(content=f"Analyze this log :\n{data}?")]},
        config
    )
print(f"Resposta:", result2["messages"][-1].content)

# Verifica estado
estado = graph.get_state(config)
print(f"\nTotal de mensagens: {len(estado.values['messages'])}")
for msg in estado.values['messages']:
    print(f"  [{msg.type}] {msg.content[:80]}...")

LLMFactory: Getting model for provider 'ollama' - metis
Resposta 1: I'm ready to analyze the events. Please provide the log data, and I'll carefully examine each event, detect anomalies, identify security events, and map my findings to MITRE ATT&CK TTPs.

Please go ahead and share the logs. I'll follow the rules you specified:

* Always base my analysis on evidence
* Do NOT hallucinate
* Be precise and technical
* Answer in English

I'm ready when you are!
LLMFactory: Getting model for provider 'ollama' - metis
Resposta: This appears to be a JSON array of security event logs from various applications and services. Here's a breakdown of the structure and content:

**Array Structure**

The data is an array of objects, where each object represents a single security event.

**Object Properties**

Each object has several properties that provide information about the event:

1. **Timestamp**: The date and time when the event occurred.
2. **ActionType**: A string describing the type of action

In [7]:
result1 = graph.invoke(
    {"messages": [HumanMessage(content='Provide a final consolidated analysis of all logs and mapping for Mitre ATT&CK')]},
    config
)
print("Resposta 1:", result1["messages"][-1].content)

LLMFactory: Getting model for provider 'ollama' - metis
Resposta 1: After analyzing the provided security event logs, I've identified various techniques and tactics used by attackers. Here's a consolidated analysis and mapping to the Mitre ATT&CK framework:

**Tactics**

1. **Initial Access**: Attackers gained access to the environment through various means, including:
	* Phishing (e.g., "MailItemsAccessed" events)
	* Exploit of vulnerabilities in applications or services (e.g., "Add delegated permission grant." events)
2. **Persistence**: Attackers maintained their presence within the environment by:
	* Creating new accounts or modifying existing ones (e.g., "AccountObjectId" changes)
	* Establishing persistence mechanisms, such as scheduled tasks or services (e.g., "Run" activity type)
3. **Privilege Escalation**: Attackers elevated their privileges to gain more control over the environment by:
	* Adding delegated permissions (e.g., "Add delegated permission grant." events)
	* Modify